# Read and audit the entire poster

**Atlas source reading copy · browser-network**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Sea-Level-Rise/blob/66065673b47f9bc5c0f6e5009ac1c924a9f5200c/content/notebooks/01_read_the_poster.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/notebooks/01_read_the_poster.ipynb` at `66065673b47f9bc5c0f6e5009ac1c924a9f5200c`; original SHA-256 `ace4a1684415597e23d3d4286b355dd3b5141df30b12e8c95be166b14e5861c6`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# Read and audit the entire poster

**Time:** 35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Trace the background → methods → results → discussion → next steps; reproduce the published tables; identify an internal discrepancy.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/01_read_the_poster.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [ ]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

## From abstract to an evidence chain

**Background.** Coastal hazards may undermine continuity of HIV services, with unequal consequences for communities already facing barriers to care.

**Methods.** The poster describes a PEPFAR site portfolio combined with Climate Central's Portfolio Analysis Tool, CoastalDEM elevation, Kopp et al. sea-level projections and Muis et al. coastal flood modeling. GIS supports spatial comparison, SaTScan supports hotspot investigation, and Power BI presents scenario tabs.

**Results.** Read Tables 1–3 as projections under assumptions, not a list of observed flood events. The reported portfolio denominator is 25,873 supported facilities in 2023. The five published city/district pairs are examples, not the entire portfolio.

**Discussion.** Exposure motivates resilience planning; it does not establish a causal estimate of HIV outcomes. Community participation, continuity of treatment and displacement are central to interpretation.

**Next steps.** The poster proposes linking monitoring indicators to estimate the services potentially affected and supporting adaptation planning. Those indicators are not supplied as public data here.

The lesson covers the final official poster. Power BI's embedded abstract has wording and numeric variants; the source audit records those rather than silently merging versions.

In [ ]:
portfolio = published.query("scope == 'portfolio'")
table = portfolio.pivot(index='year',columns='scenario',values='potentially_flooded')
display(table)
fig = px.line(portfolio,x='year',y='potentially_flooded',color='scenario',markers=True,
              color_discrete_map=COLORS,title='Published Tables 1–3: whole portfolio')
fig.update_layout(yaxis_title='Potentially flooded facilities',xaxis_title='Projection year')
show(fig)

## Recalculate before repeating a claim

The narrative reports approximately 2.8× growth under RCP 2.6 and 3.8× under RCP 8.5 from 2030 to 2100. Calculate the ratios from the tables. An inconsistency is a reason to ask the authors about versions and definitions; it is not permission to alter the source.

In [ ]:
ratio = table.loc[2100] / table.loc[2030]
audit = pd.DataFrame({'table_ratio':ratio, 'narrative_ratio':[2.8,np.nan,3.8]},index=SCENARIOS)
audit['table_percent_increase'] = 100*(audit.table_ratio-1)
display(audit.round(3))
assert table.loc[2030,'RCP 2.6'] == 108
assert table.loc[2100,'RCP 2.6'] == 321
print('RCP 2.6: 321 / 108 ≈ 2.97×, not 2.8×. RCP 8.5: 412 / 108 ≈ 3.81×.')

In [ ]:
# City and district rows are separate reporting units; choose exactly one scope.
bangkok = published.query("city == 'Bangkok' and scenario == 'RCP 2.6'")
display(bangkok.pivot(index='year',columns='scope',values=['potentially_flooded','facilities_2023']))

## Lab: write a defensible result

Calculate the 2030 and 2100 percentage for Bangkok **city** under RCP 8.5, then for its **district** row. Describe why summing those two rows would require geographic overlap information that the poster does not provide.

**Worked check:** City: 13/19 ≈ 68.4% in 2030 and 16/19 ≈ 84.2% in 2100. District: 18/24 = 75% and 21/24 = 87.5%. These small denominators produce much larger percentages than the full portfolio.

**Source-quality note:** Table 1 prints 12/137 as 6.8% in one cell; this repository retains the count and computes 8.8%. It also distinguishes the 2025 95–95–95 targets from the wider 2030 ending-AIDS agenda. Read the [UNAIDS explanation](https://www.unaids.org/en/resources/documents/2024/progress-towards-95-95-95).

**Deliverable:** A 150-word summary with background, method, one numeric result, one limitation and one next step. Cite the official poster and label your ratio as a recalculation from its tables.